# Plotly — Interactive Charts Drawn in the Browser

> 📘 **Instructor Curriculum** — M02, Data Visualization with Python

**What this notebook is:** a hands-on tour of `plotly.express`, covering line, scatter,
bar, box, violin, pie, area, 3D scatter, and sunburst charts.

**How to read it:** like the Matplotlib and Seaborn notebooks, this is a **ladder**. Most
cells repeat the cell above and change exactly one argument. The useful question at each
step is *what is different from the cell before?* Each Markdown section explains every
code cell that follows it, up to the next section.

Versions used here: **plotly 6.9.0**, **pandas 3.0.3**.

---

## The mental model — read this first

Matplotlib and seaborn draw **pixels**, in Python. Plotly does not. Plotly builds a
**JSON document** in Python, and a JavaScript library draws it in the browser.

```text
   Python                        the payload              Browser
   ------                        -----------              -------
   px.line(df, x=..., y=...)  -> {                    ->  plotly.js
                                   "data":   [ ... ],     reads the JSON
   returns a Figure                "layout": { ... }      draws the pixels
   (a dictionary, really)        }                        handles zoom / hover / legend
```

Three consequences run through this whole notebook:

1. **`fig` is data, not a picture.** You can print it, edit it, and store it.
   `fig.show()` just ships it to the browser.
2. **Interactivity is free.** Zoom, pan, hover tooltips, and clickable legends are
   already there. Nobody wrote that code — plotly.js provides it for every chart.
3. **Some work you expect Python to do happens in the browser instead.** The box-plot
   quartiles, the pie percentages, and the 3D camera are all computed there. Later
   sections show this directly in the JSON.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> This is the same split you already know: a REST service and a front end. Python is the
> service that returns a payload; plotly.js is the client that renders it. `fig.show()`
> is the call between them. That is also why a plotly chart survives outside the
> notebook — the payload is self-contained, so `fig.write_html()` produces a working
> interactive page with no Python behind it.

### `px` vs `go` — the two doors

| | `plotly.express` (`px`) | `plotly.graph_objects` (`go`) |
|---|---|---|
| Level | high | low |
| You pass | a DataFrame + column **names** | explicit arrays, one trace at a time |
| One call gives you | traces + layout + legend + colours | one trace |
| Use it for | about 95% of charts | fine control and custom combinations |

`px` writes `go` for you. Every `px` call returns a `go.Figure`, so you can always drop
down afterwards with `fig.update_layout(...)` or `fig.add_trace(...)`. That is the normal
workflow, not a workaround.

**Takeaway:** plotly separates *describing* a chart from *drawing* it. Python describes;
the browser draws.

---

## 1. Setup — install and import

| Cell | Why it exists |
|---|---|
| `pip install plotly` | Installs **plotly 6.9.0** into `pizza_env`. Its only real dependency is `narwhals`, the layer that lets plotly read pandas, polars, or pyarrow frames through one interface. |
| `pip install --upgrade pip` | Housekeeping after the install notice. Nothing to do with plotly. |
| `import plotly.express as px` | The high-level API. `px` is the universal alias. |
| `import plotly.graph_objects as go` | The low-level API. Imported here for later — no cell in this notebook uses it yet. |

Both install cells end with *"you may need to restart the kernel to use updated
packages."* That warning is real. The order that always works is: **install → restart the
kernel → import**. If an import fails right after an install, restart before debugging
anything else.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> `%pip install plotly` (with the percent sign) is the safer form inside a notebook. It
> installs into **the environment this kernel is running**. Bare `pip` works here only
> because IPython's automagic rewrites it. When a notebook and a terminal disagree about
> whether a package is installed, this is almost always why.

**Takeaway:** two imports cover the whole library — `px` for building charts, `go` for
adjusting them.

In [1]:
pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 51.4 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [6]:
import plotly.express as px
import plotly.graph_objects as go

---

## 2. Built-in datasets — `px.data`

The sample datasets are **not** top-level functions. They live in a submodule called
`px.data`:

```text
plotly.express         ->  the plotting calls      px.line, px.bar, px.scatter_3d
plotly.express.data    ->  the sample datasets     px.data.tips(), px.data.iris()
```

> ⚠️ **The error this cell started with**
>
> ```python
> tips = px.data_tips()
> # AttributeError: module 'plotly.express' has no attribute 'data_tips'
> ```
>
> It is a **dot, not an underscore**: `px.data.tips()`. `px.data` is a module you reach
> into; `tips` is a function inside it. Whenever a name is not found, this is the first
> thing to check — is the thing before the name a module or a package?
>
> The quickest way to settle it in a notebook:
> ```python
> [d for d in dir(px.data) if not d.startswith("_")]
> ```
> That prints every dataset your installed version actually has.

### The three frames used in this notebook

| Call | Shape (verified) | Used for |
|---|---|---|
| `px.data.tips()` | 244 rows × 7 cols | scatter, bar, box, violin, pie, sunburst |
| `px.data.gapminder().query("country=='India'")` | 12 rows × 8 cols | line, area |
| `px.data.iris()` | 150 rows × 6 cols | 3D scatter |

`tips` is long form — **one row per restaurant bill**: `total_bill`, `tip`, `sex`,
`smoker`, `day`, `time`, `size` (party size). The same long-form contract seaborn wants.

`.query("country=='India'")` is a pandas filter, not a plotly feature. Gapminder holds
every country; India has **12 rows**, one every five years from **1952 to 2007**.

Also available: `px.data.stocks()`, `px.data.gapminder()`, `px.data.medals_long()`,
`px.data.election()`, `px.data.carshare()`, `px.data.wind()`, `px.data.experiment()`.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> Unlike seaborn's `load_dataset`, `px.data` needs **no network**. The CSVs ship inside
> the installed package as `.csv.gz` files. Useful when you are offline or behind a
> corporate proxy — and a small lesson in packaging: bundle your fixtures with the code
> when they are small enough.

**Takeaway:** `px.data.<name>()` returns a ready long-form DataFrame. Note the dot.

In [8]:
tips = px.data.tips()
tips


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [9]:
df = px.data.gapminder().query("country=='India'")
df.head()

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
696,India,Asia,1952,37.373,372000000,546.565749,IND,356
697,India,Asia,1957,40.249,409000000,590.061996,IND,356
698,India,Asia,1962,43.605,454000000,658.347151,IND,356
699,India,Asia,1967,47.193,506000000,700.770611,IND,356
700,India,Asia,1972,50.651,567000000,724.032527,IND,356


---

## 3. Line charts — `px.line`

India's life expectancy: **12 points, 1952 → 2007, 37.4 → 64.7 years**. A near-straight
climb, so every visual change you see below comes from the argument, not the data.

| Cell | Call | The one new thing |
|---|---|---|
| A | `px.line(df, x="year", y="lifeExp")` | Baseline. The axis labels say `year` and `lifeExp` — **taken from the column names**, for free. |
| B | `+ markers=True` | Sets the trace `mode` from `"lines"` to `"lines+markers"`. The 12 real data points become visible. |
| C | `+ title="Life Expectancy of India"` | Writes `layout.title`. Nothing else changes. |

### Why `markers=True` is more than decoration

A line makes a claim: *between these two points, the value moved smoothly along this
path.* Nobody measured that. The data has 12 points, five years apart.

```text
   without markers          with markers
   ---------------          ------------
   /                        *
  /                        /
 /   looks continuous     *      you can see the 12 measurements
/                        /       and that the rest is drawn-in
                        *
```

Turning markers on separates **what was measured** from **what was drawn**. Do it by
default while you are exploring.

### `px.line` does not aggregate

This is a real difference from `sns.lineplot`, which groups rows sharing an `x`, takes
the mean, and adds a confidence band. `px.line` does none of that — it connects the rows
**in DataFrame order**. Two consequences:

1. **Sort by `x` first.** Unsorted rows give a line that jumps backwards.
2. **Duplicate `x` values give a zigzag**, not an average. If you want an average,
   aggregate in pandas before plotting.

**Common mistakes**

1. Plotting an unsorted frame and blaming plotly for the scribble.
2. Reading a line chart of a category (day, country) as a trend. A line needs a
   **naturally ordered** x — usually time.
3. Expecting a confidence band because seaborn drew one. Plotly draws exactly your rows.

**Takeaway:** `px.line` is literal — it draws your rows, in your order, and nothing else.

In [10]:
fig = px.line(df, x="year",y="lifeExp")
fig.show()

In [11]:
fig = px.line(
 df,
 x="year",
 y="lifeExp",
 markers=True
)
fig.show()

In [12]:
fig = px.line(
 df,
 x="year",
 y="lifeExp",
 title="Life Expectancy of India"
)
fig.show()

---

## 4. Scatter and the visual channels

Same core idea as seaborn's semantics: **map a column onto a visual property**.

```text
       DATA COLUMN            ARGUMENT          VISUAL CHANNEL
       -----------            --------          --------------
       sex            --->    color=     --->   marker colour  (+ legend)
       size           --->    size=      --->   marker area
```

| Cell | Call | The one new thing |
|---|---|---|
| A | `px.scatter(tips, x="total_bill", y="tip")` | Baseline. 244 dots in **one** trace. The rising cloud is the tip-to-bill relationship. |
| B | `+ color="sex"` | Splits into **two traces** — Female 87 rows, Male 157 — with a clickable legend. |
| C | `+ size="size"` | Party size (1–6) drives the marker **area**. |

### `color=` splits the data into traces — this is the whole mechanism

That one fact explains most of plotly's behaviour:

```text
color="sex"
    |
    +--> trace 0: name="Female", 87 points, colour #1   --+
    +--> trace 1: name="Male",  157 points, colour #2   --+--> legend has 2 entries
```

A trace is the unit plotly can show and hide. That is why clicking a legend entry removes
a whole group, and why grouped bars, stacked areas, and the 3D chart later all work the
same way — `color=` made separate traces.

### How `size=` is actually computed — verified from the figure JSON

```text
marker.sizemode = "area"
marker.sizeref  = max(size column) / size_max**2
                = 6 / 20**2
                = 0.015

rendered diameter (px) = sqrt(value / sizeref)

    size = 6  ->  sqrt(6 / 0.015)  = 20.0 px   <- size_max, the default cap
    size = 3  ->  sqrt(3 / 0.015)  = 14.1 px
    size = 1  ->  sqrt(1 / 0.015)  =  8.2 px
```

Read that carefully: **area is proportional to the value, not diameter.** A party of 6 is
6× a party of 1 in ink, but only 2.4× in width. This is deliberate — the eye judges the
*area* of a blob, so scaling the diameter instead would exaggerate large values by the
square. Change the cap with `size_max=30`.

**Common mistakes**

1. `color="red"` — `color` takes a **column name**. For one fixed colour use
   `color_discrete_sequence=["red"]`, or `fig.update_traces(marker_color="red")`.
2. Mapping `size=` to a category. Bigger must mean *more*. Here `size` is a genuine
   count (party size), so it is a fair use.
3. Assuming the legend order is meaningful. It follows first appearance in the data.

**Takeaway:** `color=` and `size=` are `GROUP BY` for charts — and `color=` is the one
that creates traces, which is what makes the legend interactive.

In [13]:
fig = px.scatter(
 tips,
 x="total_bill",
 y="tip"
)
fig.show()

In [14]:
fig = px.scatter(
 tips,
 x="total_bill",
 y="tip",
 color="sex"
)
fig.show()

In [15]:
fig = px.scatter(
 tips,
 x="total_bill",
 y="tip",
 size="size"
)
fig.show()

---

## 5. Bars — and the one thing plotly does *not* do

Read this section slowly. The chart below looks like the seaborn bar chart you already
know, and it means something completely different.

```python
px.bar(tips, x="day", y="total_bill")
```

244 rows, 4 days. **What does one bar mean?**

### Verified from the figure JSON

The trace carries **244 y values**, not 4. Plotly express does **not** group and does
**not** average. It ships every row to the browser and sets `barmode="relative"` —
meaning *stack them*.

```text
   seaborn barplot                        plotly express bar
   ---------------                        ------------------
   GROUP BY day                           send all 244 rows as-is
   mean(total_bill)   -> bar height       stack 244 rectangles
   bootstrap 95% CI   -> error bar        bar height = the SUM
   4 numbers leave Python                 244 numbers leave Python
```

So these bars are **sums** (verified):

| Day | Bar height (sum) | Mean bill | Rows |
|---|---|---|---|
| Sat | 1778.40 | 20.44 | 87 |
| Sun | 1627.16 | 21.41 | 76 |
| Thur | 1096.33 | 17.68 | 62 |
| Fri | **325.88** | 17.15 | 19 |

Look closely at the bars in the output: each one has faint horizontal lines running
across it. Those are the **edges of the individual stacked rectangles** — one per bill.
That is the visual tell that nothing was aggregated.

### Why this matters

By the chart, Saturday looks about **5.5×** bigger than Friday. By average bill, Saturday
is only **19%** bigger. The sum is mostly measuring **how many people came**, not how much
they spent. Both are real facts, but they answer different questions — and only one of
them is the question most people think they are asking.

If you want the average, aggregate first, in pandas:

```python
avg = tips.groupby("day", as_index=False)["total_bill"].mean()
px.bar(avg, x="day", y="total_bill")
```

That is the general plotly rule: **shape the data in pandas, then plot it.** Plotly draws;
it does not compute.

### Category order

The x axis reads **Sun, Sat, Thur, Fri**. Not alphabetical, not calendar order — plotly
uses **order of first appearance in the DataFrame**. Fix it explicitly:

```python
px.bar(tips, x="day", y="total_bill",
       category_orders={"day": ["Thur", "Fri", "Sat", "Sun"]})
```

### The other two cells

| Cell | Call | The one new thing |
|---|---|---|
| B | `+ color="sex"` | Two traces (Female 87 rows, Male 157), **stacked** on top of each other because `barmode` defaults to `"relative"`. Pass `barmode="group"` for side-by-side bars. |
| C | `x="total_bill", y="day", orientation="h"` | Horizontal bars. You swap `x` and `y` **and** set `orientation="h"` — both, not one. Useful when category names are long. |

**Common mistakes**

1. Reading a `px.bar` height as an average. It is a sum of stacked rows.
2. Forgetting `barmode="group"` and then wondering why the coloured bars are on top of
   each other rather than beside each other.
3. Trusting the category order that appears by default.
4. Sending 100k rows to `px.bar` and freezing the browser. One rectangle per row is real
   work for plotly.js — aggregate first.

**Takeaway:** **a plotly express bar is a stack of rows, so its height is a sum.** If you
want a statistic, compute it in pandas before you plot.

In [16]:
fig = px.bar(
 tips,
 x="day",
 y="total_bill"
)
fig.show()

In [17]:
fig = px.bar(
 tips,
 x="day",
 y="total_bill",
 color="sex"
)
fig.show()

In [18]:
fig = px.bar(
 tips,
 x="total_bill",
 y="day",
 orientation="h" 
)
fig.show()

---

## 6. Box and violin — one distribution per category

The bar chart above collapsed 244 bills into 4 numbers. These two charts keep the spread.

| Cell | Call | What it draws |
|---|---|---|
| A | `px.box(tips, x="day", y="total_bill")` | Five-number summary per day, plus outliers |
| B | `px.violin(tips, x="day", y="total_bill")` | A mirrored density curve — the *shape* of the distribution |

### Reading a box plot

```text
        o  o          <- outliers (beyond 1.5 x IQR from the box)
        |
      -----           <- upper fence
     |     |
     |-----|          <- Q3   (75% of bills are below this)
     |     |
     |=====|          <- MEDIAN (half the bills are below this)
     |     |          <- the box height is the IQR: the middle 50%
     |-----|          <- Q1   (25% of bills are below this)
     |     |
      -----           <- lower fence
```

The box holds the middle 50% of the data. Its height is the **IQR** — a robust measure of
spread that a single mean can never show you.

### Where are the quartiles computed?

**In the browser.** Verified: the Python trace carries all **244 raw `y` values**, and the
`q1` field is `None`. plotly.js sorts the values and computes the quartiles at draw time.
Same for the violin's density curve.

This is the mental model from the top of the notebook, made concrete:

```text
   Python  ->  "here are 244 numbers and the word 'box'"
   Browser ->  sorts, finds Q1/median/Q3, decides outliers, draws
```

It has a practical edge: the raw values travel with the chart, so hover shows real data
points, and the whole thing still works in a saved HTML file with no Python running.

### Box vs violin — the honest trade-off

| | `box` | `violin` |
|---|---|---|
| Shows | exact summary statistics | estimated shape |
| Good at | comparing medians and spread quickly | revealing two humps, skew, gaps |
| Risk | hides a two-peaked distribution completely | the smoothing can invent a shape |
| Rows needed | works with few | needs a decent number to be honest |

A box plot cannot tell you whether one hump or two produced it. That is exactly what the
violin adds. `px.violin(..., box=True, points="all")` gives you all three at once, which
is usually the right exploration setting.

**Common mistakes**

1. Calling every outlier dot an error. It is a rule (1.5 × IQR), not a judgement.
2. Comparing violins across groups with very different row counts — Friday has 19 rows,
   Saturday 87. The thin group's curve is far less trustworthy.
3. Using box plots for an audience that has never read one. They need explaining.

**Takeaway:** the box gives you the summary, the violin gives you the shape — and both
are computed in the browser from the raw rows you sent.

In [19]:
fig = px.box(
 tips,
 x="day",
 y="total_bill"
)
fig.show()

In [20]:
fig = px.violin(
 tips,
 x="day",
 y="total_bill"
)
fig.show()

---

## 7. Pie — two different ways to feed the same chart

| Cell | Call | The one new thing |
|---|---|---|
| A | `px.pie(names=["A","B","C"], values=[30,40,30])` | **No DataFrame at all.** `px` accepts plain lists. `names` are the slice labels, `values` are the slice sizes. |
| B | `px.pie(tips, names="day")` | `values` is **omitted** — so plotly counts the rows per label. |

### What cell B really does — verified

The trace carries **244 labels** and `values = None`. The browser tallies them:

```text
   Sat   87 rows   ->  128.4 degrees
   Sun   76 rows   ->  112.1 degrees
   Thur  62 rows   ->   91.5 degrees
   Fri   19 rows   ->   28.0 degrees
```

So cell B is a **count of rows per category**, drawn as a pie — seaborn's `countplot`
wearing a different hat. Whenever you leave `values` out of `px.pie`, you are counting.

### ⚠️ An honest word about pie charts

People compare **angles** badly. Saturday (87) and Sunday (76) differ by 16 degrees out of
360. In the pie you have to squint. In a bar chart the same comparison is instant, because
the eye compares **lengths against a shared baseline** very well.

```text
   as a pie: 128.4 deg vs 112.1 deg    ->  "about the same?"

   as bars:  Sat  ############# 87
             Sun  ###########   76      ->  obvious
```

Reasonable use of a pie: **2 to 4 slices, showing parts of one whole, where the exact
comparison does not matter** — a share-of-total headline. Beyond that, use a bar chart.
Cell A (3 slices, 30/40/30) is a fair use. Cell B (4 slices, two of them close) is the
case where a bar chart wins.

**Common mistakes**

1. Passing `values` when you meant to count, so slices show a sum instead of a tally.
2. More than about five slices. The small ones become unreadable noise.
3. Using a pie for values that are not parts of a single whole.

**Takeaway:** `px.pie` without `values` counts rows. And a bar chart is almost always the
more readable version of the same fact.

In [21]:
fig = px.pie(
 names=["A","B","C"],
 values=[30,40,30]
)
fig.show()

In [22]:
fig = px.pie(
 tips,
 names="day"
)
fig.show()

---

## 8. Area — a line with the region below it filled

```python
px.area(df, x="year", y="lifeExp")
```

Verified in the JSON: the trace's `fill` is not set, but `stackgroup = "1"`. That single
field is the point of `px.area`. Area charts in plotly express are **stacking charts**:

```text
   one series          ->  fills down to the baseline (what you see here)
   with color="..."    ->  each group sits ON TOP of the previous one,
                           and the top edge is the RUNNING TOTAL
```

So `px.area` is not "line plus decoration". It is the chart that says *these parts add up
to a whole*.

### ⚠️ Is it the right chart for this data?

No — and it is worth knowing why, because the chart still looks fine.

A filled area invites you to read the **area itself** as a quantity: total revenue, total
requests served, total rainfall. Life expectancy is a **rate**, not a stock. The shaded
region under it — "life-expectancy-years accumulated since 1952" — is not a real
quantity. Nothing is piling up.

| Use an area chart when | Use a line chart when |
|---|---|
| values are amounts that genuinely add up | values are rates, ratios, indexes, temperatures |
| you want to show composition over time | you want to compare levels |
| the running total is meaningful | only the current value is meaningful |

This cell is a fine way to learn the API and the wrong chart for this column. Both things
are true, and noticing the difference is the actual skill.

**Takeaway:** an area chart claims "these things add up". If they do not add up, draw a
line.

In [23]:
fig = px.area(
 df,
 x="year",
 y="lifeExp"
)
fig.show()

---

## 9. 3D scatter — how three numbers become one pixel

```python
iris = px.data.iris()
px.scatter_3d(iris, x="sepal_length", y="sepal_width", z="petal_length", color="species")
```

Verified: **3 traces** (one per species, 50 points each), trace type `scatter3d`. The
`color=` rule from section 4 is unchanged — it still splits the data into traces.

### The question this chart has to answer

Your screen is flat. A pixel has **two** coordinates. The data has **three**. So the third
number cannot survive as a position — something has to consume it. *How* it gets consumed
decides what you can and cannot trust in the picture.

### The pipeline, with real numbers

Every number below is computed from the **first row of `iris`**: sepal_length 5.1,
sepal_width 3.5, petal_length 1.4 (a setosa).

```text
   (5.1, 3.5, 1.4)          one flower, three measurements, all in cm
          |
     [1] NORMALISE          where does each value sit inside its own axis range?
          v
   (-0.254, 0.076, -0.647)  "world" point, inside a box centred on the origin
          |
     [2] VIEW               move the world so the camera sits at the origin
          v
   (0.233, -0.456, -2.641)  camera space -- the 3rd number is now DEPTH
          |
     [3] PROJECT            divide x and y by depth   <-- 3D becomes 2D HERE
          v
   (0.152, -0.417)          normalised device coords, both in -1 .. +1
          |
     [4] VIEWPORT           scale to the canvas
          v
   (403, 354)               a pixel, on a 700 x 500 chart
```

The full matrices, the source lines from `plotly.min.js` they come from, and the
arithmetic for each step are in **[`concept_3d_projection.md`](./concept_3d_projection.md)**.
Below is what each step means.

---

### Step 1 — three columns, three different units

```text
   sepal_length   4.3 .. 7.9 cm    span 3.6
   sepal_width    2.0 .. 4.4 cm    span 2.4
   petal_length   1.0 .. 6.9 cm    span 5.9
```

Three different ranges. Plotted raw, the widest column would dominate the box. So plotly
scales each axis by its own range. From `plotly.min.js`:

```js
d[i] = 1 / (dataMax[i] - dataMin[i])      // one scale factor per axis
```

Combined with the centring step that follows it, the whole of step 1 reduces to one
readable formula:

```text
   world[i] = (value - axisMid[i]) / axisRange[i] * aspect[i]
              \_________________________________/
                 "how far from the middle of this axis, as a fraction"
                        -0.5 = axis minimum,  +0.5 = axis maximum
```

Notice what happened: **the unit cancels.** Centimetres are gone. Each axis is stretched
to roughly the same length regardless of what it measures.

That is the first thing to distrust in any 3D plot. A step of 1 cm along `petal_length`
and a step of 1 cm along `sepal_width` are drawn as **different distances on screen**,
because their axes have different spans. Distance in a 3D scatter is not physical
distance — it is distance in stretched, unitless space.

`aspect[i]` puts a little of the real shape back. Verified computation:

```text
   spans            3.6      2.4      5.9
   aspect[i]  =  geometric mean of (1/span) / (1/span[i])
             ->   0.97     0.65     1.59

   check: max/min = 2.46, which is <= 4, so plotly keeps the data proportions
          (above 4 it gives up and draws a perfect 1:1:1 cube instead)
```

So the drawn box is taller in z than it is deep in y, roughly in proportion to the real
ranges. It is a compromise, not a faithful shape.

---

### Step 3 — the divide that costs you a dimension

After the view step, the point is in camera space and its third number is simply **how far
in front of the camera it is**. The projection then does:

```text
   screen_x = focal * camera_x / depth
   screen_y = focal * camera_y / depth
                                 ^^^^^
                        this division IS the 3D effect
```

That is the whole trick. Far things get divided by a bigger number, so they slide toward
the centre of the screen and shrink. It is why the cube's edges visibly converge. And it
is why **depth does not appear in the answer** — it was spent, not stored.

Plotly's default camera, verified from `plotly.min.js`:

```text
   eye        = (1.25, 1.25, 1.25)     where you are standing
   center     = (0, 0, 0)              what you are looking at
   up         = (0, 0, 1)              which way is up  -> z is the vertical axis
   projection = "perspective",  field of view 45 degrees
```

`eye` at the corner `(1.25, 1.25, 1.25)` is why **every** default plotly 3D chart looks
like you are floating above one corner of the box. Change it with:

```python
fig.update_layout(scene_camera=dict(eye=dict(x=2, y=0, z=0)))
```

---

### What the divide costs — measured on this data

Two real flowers from `iris`, projected through the default camera:

```text
   A   sepal 6.3, 3.3   petal 4.7   ->  pixel (344.6, 212.7)   depth 2.011
   B   sepal 5.5, 2.5   petal 4.0   ->  pixel (345.4, 212.7)   depth 2.348
```

**0.8 pixels apart on screen. 1.33 cm apart in the data.** They are different flowers and
the projection turned them into one dot.

Across the 150 iris rows: **29 pairs land within 3 pixels of each other, and 111 pairs
within 6.** That is not a plotly flaw — it is arithmetic. Every pixel on the screen
corresponds to a whole **line** of possible 3D points, so screen position alone can never
tell you where a point is.

```text
       eye  *------------------------------------->  one ray
                    ^          ^          ^
                  point      point      point        all three -> SAME pixel
```

### So why does rotating help?

Rotating changes `eye`. A different `eye` gives a different view matrix, a different depth
for every point, and therefore a **different set of collisions**. Points that hid each
other in one view separate in the next.

**Motion is the depth cue.** Which leads to the practical rule: a live, draggable plotly
3D chart is reasonable evidence; a **screenshot** of one is weak evidence, because the
reader cannot resolve the ambiguity you just created.

---

### When 3D is worth it, and when it is not

| Use 3D when | Avoid 3D when |
|---|---|
| you are exploring live and can rotate | the output is a static image, PDF, or slide |
| the thing genuinely is 3D — a surface, a path, a physical shape | you want to read or compare values |
| a few hundred points at most | thousands of points (occlusion becomes total) |

For **this** dataset, the 2D version is simply better:

```python
px.scatter(iris, x="petal_length", y="petal_width", color="species")
```

The petal columns separate the species almost completely, no dimension is lost, and every
point stays readable. The general rule: before reaching for a third axis, try encoding the
third variable as **colour** or **size** instead — those channels cost you nothing.

**Common mistakes**

1. Judging distance or clustering by eye in a 3D scatter. The axes are stretched and the
   depth is gone.
2. Screenshotting a 3D chart for a report. You keep the ambiguity and throw away the fix.
3. Using 3D to show three variables when colour and size would carry them without loss.
4. Sending tens of thousands of points and blaming plotly for the lag — WebGL is drawing
   every one of them, every frame.

**Takeaway:** **3D does not add a dimension to your screen — it trades one away.** The
third number survives only as the divisor in `x/depth`, and you only get it back by
rotating.

In [24]:
iris = px.data.iris()
fig = px.scatter_3d(
 iris,
 x="sepal_length",
 y="sepal_width",
 z="petal_length",
 color="species"
)
fig.show()

---

## 10. Sunburst — nesting instead of a third axis

```python
px.sunburst(tips, path=["sex", "day"])
```

The 3D chart used a third axis and paid for it. A sunburst carries a second (and third,
and fourth) variable a different way: **rings**.

`path` is the hierarchy, read **outward from the centre**:

```text
        path = ["sex", "day"]

              inner ring            outer ring
              ----------            ----------
                 sex          ->        day

                    Female (87)
                   /   |   \   \
                Sat   Thur  Sun  Fri
                 28    32    18   9

                    Male (157)
                   /   |   \   \
                Sat   Thur  Sun  Fri
                 59    30    58   10
```

### What the numbers are — verified

`values` was not passed, so plotly **counts rows**, exactly like `px.pie` in section 7.
The figure JSON carries the tree explicitly:

| id | label | parent | value |
|---|---|---|---|
| `Female` | Female | *(root)* | 87 |
| `Male` | Male | *(root)* | 157 |
| `Female/Sun` | Sun | Female | 18 |
| `Male/Sun` | Sun | Male | 58 |
| `Female/Sat` | Sat | Female | 28 |
| `Male/Sat` | Sat | Male | 59 |
| … | | | |

Every parent equals the sum of its children (87 = 18+28+32+9), and the two roots sum to
244 — the whole frame. The `id` column is why two rings can both hold a slice called
"Sun": the identity is the **path**, `Female/Sun`, not the label.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> This is the adjacency-list shape you already know from any tree in a relational
> database: `id`, `parent_id`, `value`. `px.sunburst` builds it for you from a list of
> columns; `go.Sunburst` would make you supply those three arrays yourself. It is the
> clearest example in this notebook of what `px` is actually doing on your behalf.

### Reading it

Angle = share of the parent. Click a slice and it becomes the new centre — that drill-down
is free, and it is the reason a sunburst survives more categories than a pie.

**Common mistakes**

1. Expecting a sum when you wanted a count, or vice versa. Pass `values="total_bill"` for
   money; leave it out for a tally.
2. Ordering `path` badly. `["sex","day"]` answers *"within each sex, which days?"*.
   `["day","sex"]` answers a different question. Put the variable you want to compare
   **first**.
3. Deep hierarchies. Beyond three rings the outer slices are unreadable.

**Takeaway:** a sunburst adds variables by **nesting**, not by adding an axis — so nothing
is projected away, and nothing hides behind anything else.

In [25]:
fig = px.sunburst(
 tips,
 path=["sex","day"]
)
fig.show()

---

## 11. Wrapping up

### The three libraries, side by side

| | Matplotlib | Seaborn | Plotly |
|---|---|---|---|
| Draws where | Python, into pixels | Python, into pixels | **Browser**, from JSON |
| Aggregates for you | no | **yes** (mean, CI, KDE) | **no** |
| Interactive | no | no | **yes, for free** |
| One call gives | one artist | one plot + legend + stats | traces + layout + legend |
| Best at | full control, publication images | fast statistical exploration | dashboards, sharing, exploration |

The trap this notebook keeps returning to: **plotly looks like seaborn and behaves like
matplotlib.** Seaborn computes statistics before drawing; plotly draws the rows you hand
it. When a plotly chart surprises you, the first question is always *"what did I actually
send?"*

### The pattern

```text
px.<chart>(dataframe, x="col", y="col", color="col", size="col")
```

Same shape as seaborn — you name columns, not numbers. `color=` splits the data into
traces, which is what makes legends clickable.

### Saving and sharing

```python
fig.write_html("chart.html")    # self-contained, still interactive, no Python needed
fig.write_image("chart.png")    # static image -- requires: pip install kaleido
```

`write_html` is the one that plays to plotly's strength. A colleague can open the file and
zoom, hover, and rotate it with nothing installed.

### Covered here

`px.line` (markers, title), `px.scatter` (color, size), `px.bar` (color, orientation),
`px.box`, `px.violin`, `px.pie` (lists and counts), `px.area`, `px.scatter_3d`,
`px.sunburst`, and `px.data`.

### Not covered yet

`px.histogram` (the one px chart that *does* aggregate, via `histfunc`), `facet_row` /
`facet_col`, `hover_data` and custom tooltips, `fig.update_layout` and `update_traces`,
templates and colour scales, `animation_frame`, and the `go` API that was imported at the
top but never used.

> 🧪 **Worth trying next**
>
> 1. Rebuild the section-5 bar chart on a pandas `groupby` mean and put the two charts
>    side by side. The difference is the whole lesson.
> 2. Add `barmode="group"` and `category_orders` to the coloured bar chart.
> 3. Take the 3D chart, set `scene_camera` to two different `eye` positions, and count how
>    the visible clusters change. That is section 9 in your own hands.
> 4. Compare `px.scatter(iris, x="petal_length", y="petal_width", color="species")` with
>    the 3D chart and decide honestly which one you would put in a report.

**If you remember only one thing:** plotly builds a JSON description in Python and the
browser draws it — so plotly gives you interactivity for free, but never gives you
statistics for free.